# Problem statement 2

Read a dataset from the user. (i) Use the Find-S algorithm to find the most specific hypothesis consistent with the positive examples. (ii) State the final hypothesis after processing all positive examples.

In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
try:
    df = pd.read_csv('/content/spam_dataset.csv', encoding='latin-1')
except UnicodeDecodeError:
    df = pd.read_csv('/content/spam_dataset.csv', encoding='utf-8')

if 'Is_Spam' in df.columns:
    df = df.rename(columns={'Is_Spam': 'label'})

# Drop any unnamed columns that might be present (e.g., from CSV export issues)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Display basic information about the dataset
print("Dataset head:")
display(df.head())
print("\nDataset info:")
df.info()
print("\nLabel distribution:")
display(df['label'].value_counts())


# --- Prepare Data for Find-S Algorithm ---
# df_find_s will be the main dataframe with renamed 'label' column.
df_find_s = df.copy()

print("\nDataFrame with engineered features for Find-S (using existing features):")
display(df_find_s.head())


Dataset head:


,Contains_Link,Unknown_Sender,Spelling_Errors,Urgent_Language,label
0,Yes,Yes,Low,Yes,Yes
1,Yes,Yes,High,Yes,Yes
2,No,Yes,High,Yes,No
3,Yes,Yes,High,Yes,Yes
4,No,Yes,Low,No,No



Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Contains_Link    100 non-null    object
 1   Unknown_Sender   100 non-null    object
 2   Spelling_Errors  100 non-null    object
 3   Urgent_Language  100 non-null    object
 4   label            100 non-null    object
dtypes: object(5)
memory usage: 4.0+ KB

Label distribution:


,count
label,
No,77
Yes,23



DataFrame with engineered features for Find-S (using existing features):


,Contains_Link,Unknown_Sender,Spelling_Errors,Urgent_Language,label
0,Yes,Yes,Low,Yes,Yes
1,Yes,Yes,High,Yes,Yes
2,No,Yes,High,Yes,No
3,Yes,Yes,High,Yes,Yes
4,No,Yes,Low,No,No


## (i) Using the Find-S Algorithm

The Find-S algorithm works by starting with the most specific hypothesis and generalizing it only when a positive example contradicts the current hypothesis. It ignores negative examples. The goal is to find the most specific hypothesis that covers all positive examples. Our target concept for Find-S is 'spam'.

In [2]:
# Filter for positive examples (spam), assuming 'Yes' indicates spam based on Is_Spam column
# The label in the dataset is 'Yes' for spam, 'No' for ham.
positive_examples_find_s = df_find_s[df_find_s['label'] == 'Yes'].drop(columns=['label'])
attributes = positive_examples_find_s.columns.tolist()

print(f"Number of positive examples (spam) for Find-S: {len(positive_examples_find_s)}")
print(f"Attributes for Find-S: {attributes}")

# Find-S Algorithm Implementation
def find_s_algorithm(examples, attributes):
    if examples.empty:
        return {attr: 'No positive examples' for attr in attributes}

    # Initialize the most specific hypothesis with the first positive example
    h = list(examples.iloc[0])

    # Iterate through the rest of the positive examples
    for _, example in examples.iloc[1:].iterrows():
        for i, attr_value_in_example in enumerate(example):
            # If the current hypothesis attribute value is not '?', and it doesn't match the example's value,
            # then generalize it to '?'
            if h[i] != '?' and attr_value_in_example != h[i]:
                h[i] = '?'
    return {attr: val for attr, val in zip(attributes, h)}

# Run Find-S algorithm
final_hypothesis = find_s_algorithm(positive_examples_find_s, attributes)

print("\n## (ii) Final Hypothesis after processing all positive examples (spam):")
display(final_hypothesis)


Number of positive examples (spam) for Find-S: 23
Attributes for Find-S: ['Contains_Link', 'Unknown_Sender', 'Spelling_Errors', 'Urgent_Language']

## (ii) Final Hypothesis after processing all positive examples (spam):


{'Contains_Link': 'Yes',
 'Unknown_Sender': 'Yes',
 'Spelling_Errors': '?',
 'Urgent_Language': '?'}

## Training and Comparing Conventional Models

The user also requested training two models, using a train-test split, and comparing them. The Find-S algorithm itself generates a specific hypothesis, rather than a predictive model directly evaluated with standard metrics like accuracy in a train/test setup. However, we can use the learned Find-S hypothesis to *classify* new instances and then compare its performance with a traditional classifier.

For the second model, I will use a Multinomial Naive Bayes classifier, which is a common and effective choice for text classification tasks like spam detection. I will use TF-IDF vectorization to convert the text data into numerical features suitable for this classifier.

In [3]:
# --- Prepare Data for Conventional Classifier ---

# Map labels to numerical values: 'No' -> 0 (ham), 'Yes' -> 1 (spam)
df['label_numeric'] = df['label'].map({'No': 0, 'Yes': 1})

# Features are the categorical columns, excluding the label
X = df.drop(columns=['label', 'label_numeric'])
y = df['label_numeric']

# Split the dataset into training and testing sets (90-10 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

print(f"Train set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")

# Apply One-Hot Encoding for categorical features
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_encoded = ohe.fit_transform(X_train)
X_test_encoded = ohe.transform(X_test)

# --- Train and Evaluate Decision Tree Classifier (suitable for categorical features) ---
print("\n--- Decision Tree Classifier ---")
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train_encoded, y_train)

y_pred_dt = dt_classifier.predict(X_test_encoded)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
report_dt = classification_report(y_test, y_pred_dt, target_names=['ham', 'spam'])

print(f"Accuracy (Decision Tree Classifier): {accuracy_dt:.4f}")
print("Classification Report (Decision Tree Classifier):\n", report_dt)

# Store accuracy for comparison
accuracy_mnb = accuracy_dt # Renaming for consistency with original notebook's comparison section


Train set size: 90 samples
Test set size: 10 samples

--- Decision Tree Classifier ---
Accuracy (Decision Tree Classifier): 1.0000
Classification Report (Decision Tree Classifier):
               precision    recall  f1-score   support

         ham       1.00      1.00      1.00         8
        spam       1.00      1.00      1.00         2

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



## Comparison of Models

Comparing the Find-S algorithm directly with a predictive classifier like Naive Bayes is not straightforward because they serve different purposes. Find-S aims to find the *most specific hypothesis* for positive examples, while a classifier aims for high generalization accuracy on unseen data.

However, to fulfill the request of showing which one works better, we can adapt the Find-S hypothesis into a simple classifier. This 'Find-S classifier' will predict 'spam' if an instance perfectly matches the learned specific hypothesis (where '?' matches any value), and 'ham' otherwise. We will evaluate this on the same test set using the *engineered categorical features*.

**Important Note:** This comparison is illustrative and not a rigorous scientific evaluation, as Find-S is a concept learning algorithm, not designed for general predictive accuracy on a test set in the same way as a probabilistic classifier.

In [4]:
# --- Evaluate the Find-S 'Classifier' ---

# Create a function to classify based on the Find-S hypothesis
def classify_with_find_s_hypothesis(instance_features, hypothesis):
    for attr, hyp_val in hypothesis.items():
        # Use .get() to safely access instance_features, returning None if key not found
        instance_val = instance_features.get(attr)
        if hyp_val == '?': # '?' matches any value
            continue
        elif instance_val != hyp_val: # Mismatch if not '?' and not equal
            return 0 # Predict 'ham' (negative)
    return 1 # Predict 'spam' (positive) if no contradictions found

# Prepare test data with the same categorical features used for training
# X_test already contains the categorical features, so no extra feature extraction is needed.
find_s_test_features = X_test.copy()

y_pred_find_s = find_s_test_features.apply(
    lambda row: classify_with_find_s_hypothesis(row.to_dict(), final_hypothesis),
    axis=1
)

accuracy_find_s = accuracy_score(y_test, y_pred_find_s)
report_find_s = classification_report(y_test, y_pred_find_s, target_names=['ham', 'spam'])

print("\n--- Find-S 'Classifier' Performance ---")
print(f"Accuracy (Find-S Classifier): {accuracy_find_s:.4f}")
print("Classification Report (Find-S Classifier):\n", report_find_s)

print("\n--- Which one works better? ---")
print(f"Decision Tree Accuracy: {accuracy_mnb:.4f}") # Changed from Multinomial Naive Bayes
print(f"Find-S 'Classifier' Accuracy: {accuracy_find_s:.4f}")

if accuracy_mnb > accuracy_find_s:
    print("\nBased on these evaluations, the Decision Tree classifier significantly outperforms the Find-S 'classifier'. ")
else:
    print("\nBased on these evaluations, the Find-S 'classifier' performed comparably or better.")



--- Find-S 'Classifier' Performance ---
Accuracy (Find-S Classifier): 1.0000
Classification Report (Find-S Classifier):
               precision    recall  f1-score   support

         ham       1.00      1.00      1.00         8
        spam       1.00      1.00      1.00         2

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10


--- Which one works better? ---
Decision Tree Accuracy: 1.0000
Find-S 'Classifier' Accuracy: 1.0000

Based on these evaluations, the Find-S 'classifier' performed comparably or better.
